# Regime B: 3-Shot Prompting Evaluation (Pengujian Adil Model Pembanding)
### Evaluasi Ulang Model yang Gagal di Regime A Menggunakan Strict System Prompt + 3 Few-Shot BSS Examples

**Latar Belakang:**
Pada Regime A (Zero-Shot), model cloud (Claude Opus 5, GPT-5.6-Terra, dll) dan model open-source (Llama, Phi, Gemma, dll)
mendapatkan skor Strict AST 45.54% -- mereka gagal memanggil API BSS karena **tidak pernah melihat contoh format tool call**.

**Tujuan Regime B:**
Memberikan **kesempatan yang adil** kepada model pembanding dengan:
1. [OK] System prompt KETAT yang mewajibkan format `<tool_call>JSON</tool_call>`
2. [OK] 3 contoh Few-Shot nyata dari API BSS TelkomNusa
3. [OK] strict instruction verification rules

**Model yang TIDAK diuji ulang** (sudah memiliki keunggulan SFT):
- âŒ TelkomNusa-SLM-3B (sudah SFT -> 80.20%)
- âŒ TelkomNusa-LLM-7B (sudah SFT -> 78.22%)
- âŒ Qwen2.5-3B-Base-Zero (sudah ada: 55.45%)
- âŒ Qwen2.5-3B-Base-CoT (sudah ada: 59.41%)


## Section 0: Setup Dependencies & API Keys


In [ ]:
# 1. Pasang dependensi yang diperlukan tanpa konflik versi
!pip install -q peft accelerate bitsandbytes huggingface_hub openai google-generativeai requests nltk rouge-score

import os, sys, re, ast, json, time, gc, torch, requests, math
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
from typing import List, Dict, Any, Tuple, Optional
from sklearn.metrics import f1_score, accuracy_score
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from openai import OpenAI

try:
    import transformers.cache_utils as _tcu
    if not hasattr(_tcu, "SlidingWindowCache"):
        _tcu.SlidingWindowCache = getattr(_tcu, "DynamicCache", object)
        print("[OK] Patch SlidingWindowCache aktif.")
except Exception:
    pass
try:
    import transformers.utils.import_utils as _tfu
    _tfu.is_torchvision_available = lambda: False
except Exception:
    pass

import nltk
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from rouge_score import rouge_scorer

smoothie = SmoothingFunction().method1
rouge_evaluator = rouge_scorer.RougeScorer(['rouge1', 'rougeL'], use_stemmer=False)

# API KEYS (SAMA DENGAN REGIME A)
JUSTWOKER_API_KEY  = os.environ.get("JUSTWOKER_API_KEY", "YOUR_API_KEY_HERE")
BLUESMINDS_API_KEY = os.environ.get("BLUESMINDS_API_KEY", "YOUR_API_KEY_HERE")
OPENROUTER_API_KEY = os.environ.get("OPENROUTER_API_KEY", "YOUR_OPENROUTER_API_KEY_HERE")
GEMINI_API_KEY     = os.environ.get("GEMINI_API_KEY", "YOUR_GEMINI_API_KEY_HERE")

client_justwoker = None
if JUSTWOKER_API_KEY:
    try:
        client_justwoker = OpenAI(api_key=JUSTWOKER_API_KEY, base_url="https://api.justwoker.icu/v1")
        print("[OK] JustWoker Client Siap!")
    except Exception as e:
        print(f"[WARNING] JustWoker: {e}")

client_bluesminds = None
if BLUESMINDS_API_KEY:
    try:
        client_bluesminds = OpenAI(api_key=BLUESMINDS_API_KEY, base_url="https://api.bluesminds.com/v1")
        print("[OK] Bluesminds Client Siap!")
    except Exception as e:
        print(f"[WARNING] Bluesminds: {e}")

client_openrouter = None
if OPENROUTER_API_KEY:
    try:
        client_openrouter = OpenAI(api_key=OPENROUTER_API_KEY, base_url="https://openrouter.ai/api/v1")
        print("[OK] OpenRouter Client Siap!")
    except Exception as e:
        print(f"[WARNING] OpenRouter: {e}")

if GEMINI_API_KEY:
    try:
        import google.generativeai as genai
        genai.configure(api_key=GEMINI_API_KEY)
        print("[OK] Google Gemini Client Siap!")
    except Exception as e:
        print(f"[WARNING] Gemini: {e}")

if torch.cuda.is_available():
    print(f"[OK] GPU: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB)")
else:
    print("[INFO] Mode CPU / Cloud API Aktif")


## Section 1: Pemuatan Dataset Held-Out Benchmark


In [ ]:
POSSIBLE_BENCH_PATHS = [
    "/kaggle/input/datasets/mrislantristansyah/id-telagentbench/dataset_telagentbench_indonesia_clean",
    "/kaggle/input/id-telagentbench/dataset_telagentbench_indonesia_clean",
    "/kaggle/input/id-telagentbench",
    "/kaggle/input/dataset-telagentbench-indonesia-clean",
    "/kaggle/input/dataset_telagentbench_indonesia_clean",
    "/content/dataset_telagentbench_indonesia_clean",
    "/content/dataset_telagentbench_indonesia_clean/dataset_telagentbench_indonesia_clean",
    "/content/id-telagentbench/dataset_telagentbench_indonesia_clean",
    "dataset_telagentbench_indonesia_clean",
    "../dataset_telagentbench_indonesia_clean"
]
DATA_DIR = None
for p in POSSIBLE_BENCH_PATHS:
    if os.path.exists(p):
        if os.path.exists(os.path.join(p, "TelAgent_Action")):
            DATA_DIR = p; break
        elif os.path.exists(os.path.join(p, "dataset_telagentbench_indonesia_clean", "TelAgent_Action")):
            DATA_DIR = os.path.join(p, "dataset_telagentbench_indonesia_clean"); break
if DATA_DIR is None:
    for s_root in ["/kaggle/input", "/content", "."]:
        if os.path.exists(s_root):
            for root, dirs, files in os.walk(s_root):
                if "TelAgent_Action" in dirs:
                    DATA_DIR = root; break
            if DATA_DIR: break
if DATA_DIR is None:
    raise FileNotFoundError("Folder benchmark 'TelAgent_Action' tidak ditemukan!")

print(f"[OK] Folder Benchmark: {DATA_DIR}")

ACTION_DIR = os.path.join(DATA_DIR, "TelAgent_Action")
SIMPLE_FILE = os.path.join(ACTION_DIR, "telco_simple.json")
SIMPLE_ANS_FILE = os.path.join(ACTION_DIR, "possible_answer", "telco_simple.json")
MULTI_FILE = os.path.join(ACTION_DIR, "telco_multi_turn_stateless_base.json")
MULTI_ANS_FILE = os.path.join(ACTION_DIR, "possible_answer", "telco_multi_turn_stateless_base.json")
TOOLS_FILE = os.path.join(ACTION_DIR, "250918_telcoFC_tools.json")

TELCO_TOOLS = []
with open(TOOLS_FILE, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            try: TELCO_TOOLS.append(json.loads(line.strip()))
            except: pass
if not TELCO_TOOLS:
    with open(TOOLS_FILE, "r", encoding="utf-8") as f:
        TELCO_TOOLS = json.load(f)

FORMATTED_TOOLS = []
for t in TELCO_TOOLS:
    if "type" in t and t["type"] == "function":
        FORMATTED_TOOLS.append(t)
    else:
        FORMATTED_TOOLS.append({"type": "function", "function": t})

simple_instances = []
with open(SIMPLE_FILE, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip(): simple_instances.append(json.loads(line))

simple_answers = {}
with open(SIMPLE_ANS_FILE, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            item = json.loads(line)
            simple_answers[item["id"]] = item["ground_truth"]

multi_instances = []
with open(MULTI_FILE, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip(): multi_instances.append(json.loads(line))

multi_answers = {}
if os.path.exists(MULTI_ANS_FILE):
    with open(MULTI_ANS_FILE, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                item = json.loads(line)
                multi_answers[item["id"]] = item["ground_truth"]

print(f"[OK] Dataset: {len(simple_instances)} Single-Turn + {len(multi_instances)} Multi-Turn = {len(simple_instances)+len(multi_instances)} soal")


## Section 2: Definisi System Prompt KETAT + 3 Few-Shot BSS Examples (Kunci Regime B)


In [ ]:
# REGIME B: STRICT SYSTEM PROMPT + 3 FEW-SHOT EXAMPLES BSS TELAGENTBENCH

STRICT_SYSTEM_PROMPT = """Anda adalah agen API resmi Customer Care TelkomNusa Indonesia.

ATURAN EKSEKUSI MUTLAK:
1. Jika pelanggan meminta transaksi (cek pulsa, kuota, tagihan, bayar tunggakan,
   aktivasi/pembatalan paket, roaming, refill kupon, info profil, dll),
   Anda WAJIB memanggil tool API yang sesuai dari daftar tools.
2. Format pemanggilan tool WAJIB menggunakan tag XML:
   <tool_call>{"name": "NAMA_API", "arguments": {"param": "value"}}</tool_call>
3. DILARANG KERAS menjawab dengan kalimat percakapan / instruksi manual
   untuk pertanyaan transaksional yang sudah ada tool-nya.
4. Jika tidak ada tool yang relevan (hanya sapaan/keluhan umum), barulah jawab dengan teks biasa.
5. Ekstrak parameter (nomor telepon, tanggal, ID produk, kode negara)
   langsung dari pesan pelanggan."""

# 3 Contoh Few-Shot BSS TelkomNusa
FEW_SHOT_EXAMPLES = [
    # Contoh 1: Cek Tunggakan Tagihan
    {
        "role": "user",
        "content": "Mohon periksa tunggakan tagihan untuk nomor 0109998888."
    },
    {
        "role": "assistant",
        "content": '<tool_call>{"name": "GET__rate_unpaid-bills", "arguments": {"svcMgmtNum": "0109998888"}}</tool_call>'
    },
    # Contoh 2: Cek Kuota Data
    {
        "role": "user",
        "content": "Tolong cek sisa kuota internet saya, nomor 0101234567."
    },
    {
        "role": "assistant",
        "content": '<tool_call>{"name": "GET__rate_current-data-limits", "arguments": {"svcMgmtNum": "0101234567"}}</tool_call>'
    },
    # Contoh 3: Cek Tagihan Keluarga
    {
        "role": "user",
        "content": "Tampilkan tagihan anggota keluarga nomor 01055556666 yang terdaftar di nomor utama 01077778888."
    },
    {
        "role": "assistant",
        "content": '<tool_call>{"name": "GET__rate_family_bills", "arguments": {"familySvcMgmtNum": "01055556666", "svcMgmtNum": "01077778888"}}</tool_call>'
    }
]

def build_fewshot_messages(user_messages: list) -> list:
    """Membangun context messages dengan Strict Prompt + 3 Few-Shot BSS."""
    messages = [{"role": "system", "content": STRICT_SYSTEM_PROMPT}]
    messages.extend(FEW_SHOT_EXAMPLES)
    messages.extend(user_messages)
    return messages

print("[OK] Regime B Strict System Prompt & 3 Few-Shot BSS Examples Terdefinisi!")
print(f"System Prompt: {len(STRICT_SYSTEM_PROMPT)} karakter")
print(f"Few-Shot Examples: {len(FEW_SHOT_EXAMPLES) // 2} pasang contoh Q&A BSS")


## Section 3: Model Registry Regime B


In [ ]:
# MODEL YANG DIUJI ULANG DI REGIME B

REGIME_B_MODELS = {
    # ---- GPU LOKAL (OPEN SLM) ----
    "Llama-3.2-3B-FewShot": {
        "platform": "local_hf",
        "model_id": "unsloth/Llama-3.2-3B-Instruct-bnb-4bit",
        "params": "3B",
        "category": "Open SLM + FewShot",
        "mode": "3-Shot Prompting (Regime B)",
        "active": True
    },
    "Phi-4-mini-FewShot": {
        "platform": "local_hf",
        "model_id": "microsoft/Phi-4-mini-instruct",
        "params": "3.8B",
        "category": "Open SLM + FewShot",
        "mode": "3-Shot Prompting (Regime B)",
        "active": True
    },

    # ---- OPENROUTER (OPEN-SOURCE 7B-30B & MOE) ----
    "Gemma-4-26B-FewShot": {
        "platform": "api_openrouter",
        "model_id": "google/gemma-4-26b-a4b-it:free",
        "params": "26B (3.8B Active)",
        "category": "Open MoE + FewShot",
        "mode": "3-Shot Prompting (Regime B)",
        "active": True
    },
    "GLM-5.2-FewShot": {
        "platform": "api_openrouter",
        "model_id": "z-ai/glm-5.2:free",
        "params": "18.6B Active",
        "category": "Open Reasoning + FewShot",
        "mode": "3-Shot Prompting (Regime B)",
        "active": True
    },
    "MiniMax-M3-FewShot": {
        "platform": "api_openrouter",
        "model_id": "minimax/minimax-m3:free",
        "params": "MoE Agentic",
        "category": "Open Agentic + FewShot",
        "mode": "3-Shot Prompting (Regime B)",
        "active": True
    },
    "Nemotron-3.5-FewShot": {
        "platform": "api_openrouter",
        "model_id": "nvidia/nemotron-3.5-lightning:free",
        "params": "30B-A3B",
        "category": "Open MoE + FewShot",
        "mode": "3-Shot Prompting (Regime B)",
        "active": True
    },

    # ---- PROPRIETARY CLOUD MODELS ----
    "Claude-Opus-5-FewShot": {
        "platform": "api_justwoker",
        "model_id": "claude-opus-5",
        "params": "Proprietary Cloud",
        "category": "Anthropic + FewShot",
        "mode": "3-Shot Prompting (Regime B)",
        "active": True
    },
    "GPT-5.6-Terra-FewShot": {
        "platform": "api_bluesminds",
        "model_id": "gpt-5.6-terra",
        "params": "Proprietary Cloud",
        "category": "OpenAI + FewShot",
        "mode": "3-Shot Prompting (Regime B)",
        "active": True
    },
    "GPT-4o-FewShot": {
        "platform": "api_bluesminds",
        "model_id": "gpt-4o",
        "params": "Proprietary Cloud",
        "category": "OpenAI + FewShot",
        "mode": "3-Shot Prompting (Regime B)",
        "active": True
    },
    "Gemini-2.5-Flash-FewShot": {
        "platform": "api_gemini",
        "model_id": "gemini-2.5-flash",
        "params": "Cloud API",
        "category": "Google + FewShot",
        "mode": "3-Shot Prompting (Regime B)",
        "active": True
    }
}

active = [k for k, v in REGIME_B_MODELS.items() if v.get("active", False)]
print(f"[OK] Regime B Registry: {len(active)} Model Siap Diuji Ulang:")
for i, m in enumerate(active, 1):
    info = REGIME_B_MODELS[m]
    print(f"{i:2d}. [{info['category']:25s}] {m:30s} ({info['params']} via {info['platform']})")


## Section 4: Universal Parser & Evaluator (Diperkuat Bare-JSON Fallback)


In [ ]:
def extract_tool_calls(response_text: str) -> List[Dict[str, Any]]:
    calls = []
    if not response_text: return calls
    xml_pattern = r"<tool_call>(.*?)</tool_call>"
    xml_matches = re.findall(xml_pattern, response_text, re.DOTALL)
    if xml_matches:
        for match in xml_matches:
            try:
                parsed = json.loads(match.strip())
                if isinstance(parsed, dict) and "name" in parsed: calls.append(parsed)
            except: pass
        if calls: return calls
    json_block_pattern = r"```(?:json)?\s*([\{\[].*?[\}\]])\s*```"
    json_matches = re.findall(json_block_pattern, response_text, re.DOTALL)
    for jm in json_matches:
        try:
            parsed = json.loads(jm.strip())
            if isinstance(parsed, dict) and "name" in parsed: calls.append(parsed)
            elif isinstance(parsed, list):
                for item in parsed:
                    if isinstance(item, dict) and "name" in item: calls.append(item)
        except: pass
    if calls: return calls
    try:
        raw_p = json.loads(response_text.strip())
        if isinstance(raw_p, dict) and "name" in raw_p: return [raw_p]
        if isinstance(raw_p, list):
            for item in raw_p:
                if isinstance(item, dict) and "name" in item: calls.append(item)
            if calls: return calls
    except: pass
    fn_pattern = r"([A-Za-z0-9_]+__[\w\-]+)\((.*?)\)"
    fn_matches = re.findall(fn_pattern, response_text)
    for fname, args_str in fn_matches:
        arg_dict = {}
        kv_pairs = re.findall(r"([a-zA-Z0-9_]+)\s*=\s*(['\"]?)(.*?)\2(?:,|$)", args_str)
        for k, _, v in kv_pairs:
            try: arg_dict[k] = ast.literal_eval(v)
            except: arg_dict[k] = v
        calls.append({"name": fname, "arguments": arg_dict})
    bare_json_pattern = r'\{"name"\s*:\s*"([^"]+)"\s*,\s*"arguments"\s*:\s*(\{[^}]+\})\}'
    bare_matches = re.findall(bare_json_pattern, response_text)
    if not calls and bare_matches:
        for fname, args_str in bare_matches:
            try:
                args = json.loads(args_str)
                calls.append({"name": fname, "arguments": args})
            except: pass
    return calls

def normalize_ground_truth(gt_raw: Any) -> List[Dict[str, Any]]:
    calls = []
    if not gt_raw: return calls
    items = []
    if isinstance(gt_raw, list):
        for item in gt_raw:
            if isinstance(item, list): items.extend(item)
            else: items.append(item)
    elif isinstance(gt_raw, dict): items.append(gt_raw)
    for item in items:
        if isinstance(item, dict):
            if "name" in item: calls.append({"name": item["name"], "arguments": item.get("arguments", {})})
            else:
                for fname, fargs in item.items(): calls.append({"name": fname, "arguments": fargs})
    return calls

def compute_nlg_metrics(candidate_text: str, reference_text: str) -> Tuple[float, float]:
    if not candidate_text or not reference_text: return 0.0, 0.0
    cand_tokens = candidate_text.strip().lower().split()
    ref_tokens = [reference_text.strip().lower().split()]
    try: b4 = sentence_bleu(ref_tokens, cand_tokens, weights=(0.25,0.25,0.25,0.25), smoothing_function=smoothie)*100
    except: b4 = 0.0
    try:
        r_scores = rouge_evaluator.score(reference_text, candidate_text)
        r_l = r_scores['rougeL'].fmeasure * 100
    except: r_l = 0.0
    return round(b4,2), round(r_l,2)

def evaluate_instance(pred_calls, gold_calls):
    if not pred_calls and not gold_calls: return True, True, True, True, "Correct (No Tool)"
    if not pred_calls and gold_calls: return False, False, False, False, "No Tool Emitted"
    if pred_calls and not gold_calls: return False, False, False, True, "Spurious Tool Call"
    pred_tool = pred_calls[0].get("name","").strip().lower()
    gold_tool = gold_calls[0].get("name","").strip().lower()
    if pred_tool != gold_tool:
        return False, False, False, True, f"Wrong Tool ({pred_calls[0].get('name','')} vs {gold_calls[0].get('name','')})"
    pred_args = pred_calls[0].get("arguments", {})
    if isinstance(pred_args, str):
        try: pred_args = json.loads(pred_args)
        except: pred_args = {}
    gold_args = gold_calls[0].get("arguments", {})
    if isinstance(gold_args, str):
        try: gold_args = json.loads(gold_args)
        except: gold_args = {}
    for k, g_val in gold_args.items():
        if k not in pred_args:
            if isinstance(g_val, list) and (None in g_val or "" in g_val): continue
            return True, False, False, True, f"Missing Param: {k}"
        p_val_str = str(pred_args[k]).strip().lower()
        if isinstance(g_val, list):
            allowed = [str(x).strip().lower() for x in g_val if x is not None]
            if allowed and p_val_str not in allowed:
                return True, False, False, True, f"Wrong Value {k}"
        else:
            if p_val_str != str(g_val).strip().lower():
                return True, False, False, True, f"Wrong Value {k}"
    return True, True, True, True, "Correct"

print("[OK] Universal Parser & Evaluator Siap!")


## Section 5: € Eksekusi Regime B -- Inferensi 3-Shot Prompting


In [ ]:
BASE_OUTPUT_DIR = "/kaggle/working" if os.path.exists("/kaggle/working") else "/content"

all_regimeB_summary = []

for model_name, cfg in REGIME_B_MODELS.items():
    if not cfg.get("active", False): continue
    print("="*85)
    print(f"[*] REGIME B | EVALUASI: {model_name} [{cfg['category']}] via {cfg['platform']}")
    print("="*85)

    platform = cfg["platform"]
    model_records = []
    eval_start_time = time.time()
    total_tokens = 0
    tokenizer = None
    model = None

    try:
        if platform == "local_hf":
            print(f"[*] Memuat Model: {cfg['model_id']} (4-Bit)...")
            tokenizer = AutoTokenizer.from_pretrained(cfg["model_id"], trust_remote_code=True)
            if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token
            bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
            try:
                model = AutoModelForCausalLM.from_pretrained(cfg["model_id"], quantization_config=bnb_config, device_map="auto", trust_remote_code=True)
            except Exception as e_load:
                print(f"[!] Fallback native load...")
                model = AutoModelForCausalLM.from_pretrained(cfg["model_id"], quantization_config=bnb_config, device_map="auto", trust_remote_code=False)
            model.eval()
        elif platform == "api_justwoker":
            if client_justwoker is None: print(f"[SKIP] {model_name}"); continue
        elif platform == "api_bluesminds":
            if client_bluesminds is None: print(f"[SKIP] {model_name}"); continue
        elif platform == "api_openrouter":
            if client_openrouter is None: print(f"[SKIP] {model_name}"); continue
        elif platform == "api_gemini":
            if not GEMINI_API_KEY: print(f"[SKIP] {model_name}"); continue
    except Exception as e_init:
        print(f"âŒ GAGAL MEMUAT {model_name}: {e_init}"); continue

    test_suite = [("Single-Turn", simple_instances, simple_answers), ("Multi-Turn", multi_instances, multi_answers)]

    for set_name, instances, answers in test_suite:
        for inst in tqdm(instances, desc=f"{model_name} [{set_name}]"):
            q_id = inst["id"]
            q_msgs = inst["question"]
            gold_call_raw = answers.get(q_id, [])
            gold_calls = normalize_ground_truth(gold_call_raw)
            ref_answer_text = json.dumps(gold_calls) if gold_calls else "Tidak memerlukan tool"

            user_part = []
            if set_name == "Single-Turn":
                user_msg = q_msgs[0] if isinstance(q_msgs, list) else q_msgs
                if isinstance(user_msg, list): user_part.extend(user_msg)
                else: user_part.append(user_msg)
            else:
                for turn in q_msgs:
                    if isinstance(turn, list): user_part.extend(turn)
                    else: user_part.append(turn)

            messages = build_fewshot_messages(user_part)

            resp_text = ""
            lat = 0.0
            gen_tok = 0

            if platform == "local_hf":
                prompt_text = tokenizer.apply_chat_template(messages, tools=FORMATTED_TOOLS, tokenize=False, add_generation_prompt=True)
                inputs = tokenizer(prompt_text, return_tensors="pt").to("cuda")
                t0 = time.time()
                with torch.no_grad():
                    out = model.generate(**inputs, max_new_tokens=200, do_sample=False, use_cache=True, pad_token_id=tokenizer.eos_token_id)
                lat = time.time() - t0
                gen_tok = out.shape[1] - inputs.input_ids.shape[1]
                total_tokens += gen_tok
                resp_text = tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=False)

            elif platform in ["api_justwoker", "api_bluesminds", "api_openrouter"]:
                if platform == "api_justwoker": client = client_justwoker
                elif platform == "api_bluesminds": client = client_bluesminds
                else: client = client_openrouter

                t0 = time.time()
                try:
                    res = client.chat.completions.create(
                        model=cfg["model_id"],
                        messages=messages,
                        tools=FORMATTED_TOOLS,
                        temperature=0.0
                    )
                    lat = time.time() - t0
                    msg = res.choices[0].message
                    if msg.tool_calls:
                        resp_text = json.dumps([{"name": tc.function.name, "arguments": json.loads(tc.function.arguments)} for tc in msg.tool_calls])
                    else:
                        resp_text = msg.content or ""
                    gen_tok = res.usage.completion_tokens if res.usage else 30
                    total_tokens += gen_tok
                except Exception as e_api:
                    lat = time.time() - t0
                    resp_text = f"API_ERROR: {e_api}"

            elif platform == "api_gemini":
                import google.generativeai as genai
                t0 = time.time()
                try:
                    g_model = genai.GenerativeModel(cfg["model_id"])
                    full_context = STRICT_SYSTEM_PROMPT + "\n\n"
                    for ex in FEW_SHOT_EXAMPLES:
                        full_context += f"[{ex['role'].upper()}]: {ex['content']}\n"
                    last_query = messages[-1]["content"] if isinstance(messages[-1], dict) else str(messages[-1])
                    full_context += f"\n[USER]: {last_query}\n[ASSISTANT]:"
                    res = g_model.generate_content(full_context)
                    lat = time.time() - t0
                    resp_text = res.text if hasattr(res, "text") else ""
                    gen_tok = 30
                    total_tokens += gen_tok
                except Exception as e_gem:
                    lat = time.time() - t0
                    resp_text = f"GEMINI_ERROR: {e_gem}"

            pred_calls = extract_tool_calls(resp_text)
            t_acc, a_match, s_ast, f_val, err_cat = evaluate_instance(pred_calls, gold_calls)
            b4_score, rl_score = compute_nlg_metrics(resp_text, ref_answer_text)

            model_records.append({
                "id": q_id, "type": set_name,
                "gold_tool": gold_calls[0]["name"] if gold_calls else "NONE",
                "pred_tool": pred_calls[0]["name"] if pred_calls else "NONE",
                "gold_args": json.dumps(gold_calls[0]["arguments"]) if gold_calls else "{}",
                "pred_args": json.dumps(pred_calls[0]["arguments"]) if pred_calls else "{}",
                "raw_response": resp_text,
                "tool_accuracy": int(t_acc), "arg_match": int(a_match),
                "strict_ast": int(s_ast), "format_valid": int(f_val),
                "bleu4": b4_score, "rougeL": rl_score,
                "error_category": err_cat, "latency_s": lat, "tokens_gen": gen_tok
            })

    df_m = pd.DataFrame(model_records)
    total_dur = time.time() - eval_start_time
    clean_mname = model_name.replace("/", "_")
    summary_entry = {
        "Model Name": model_name, "Platform": cfg["platform"], "Params": cfg["params"],
        "Mode": cfg["mode"],
        "Tool Acc (%)": round(df_m["tool_accuracy"].mean()*100, 2),
        "Macro F1 (%)": round(f1_score(df_m["gold_tool"], df_m["pred_tool"], average="macro", zero_division=0)*100, 2),
        "Arg Match (%)": round(df_m["arg_match"].mean()*100, 2),
        "Strict AST (%)": round(df_m["strict_ast"].mean()*100, 2),
        "Format Valid (%)": round(df_m["format_valid"].mean()*100, 2),
        "BLEU-4": round(df_m["bleu4"].mean(), 2),
        "ROUGE-L": round(df_m["rougeL"].mean(), 2),
        "Latency (s)": round(df_m["latency_s"].mean(), 2),
        "Throughput (tok/s)": round(total_tokens / max(0.01, total_dur), 2)
    }
    all_regimeB_summary.append(summary_entry)

    json_path = os.path.join(BASE_OUTPUT_DIR, f"RegimeB_{clean_mname}_predictions.json")
    csv_path = os.path.join(BASE_OUTPUT_DIR, f"RegimeB_{clean_mname}_predictions.csv")
    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(model_records, f, ensure_ascii=False, indent=2)
    df_m.to_csv(csv_path, index=False)
    print(f"[OK] Log: {json_path}")

    if model is not None: del model, tokenizer
    torch.cuda.empty_cache(); gc.collect()

print("\n[OK] REGIME B SELESAI! Seluruh model pembanding telah diuji dengan 3-Shot Prompting (Regime B)!")


## Section 6:  Tabel Perbandingan REGIME A vs REGIME B + Model Kita (Head-to-Head Final)


In [ ]:
regime_a_results = [
    {"Model Name": "TelkomNusa-SLM-3B",       "Regime": "A (Zero-Shot SFT)",    "Params": "3B",              "Tool Acc (%)": 85.15, "Macro F1 (%)": 74.22, "Arg Match (%)": 80.20, "Strict AST (%)": 80.20, "Format Valid (%)": 100.00, "BLEU-4":  2.93, "ROUGE-L": 38.81, "Latency (s)": 11.71, "Throughput (tok/s)":  4.69},
    {"Model Name": "TelkomNusa-LLM-7B",       "Regime": "A (Zero-Shot SFT)",    "Params": "7B",              "Tool Acc (%)": 81.19, "Macro F1 (%)": 67.59, "Arg Match (%)": 78.22, "Strict AST (%)": 78.22, "Format Valid (%)":  99.01, "BLEU-4":  2.88, "ROUGE-L": 38.66, "Latency (s)": 14.29, "Throughput (tok/s)":  3.78},
    {"Model Name": "Qwen2.5-3B-Base-Zero",    "Regime": "A (Zero-Shot)",        "Params": "3B",              "Tool Acc (%)": 61.39, "Macro F1 (%)": 47.26, "Arg Match (%)": 55.45, "Strict AST (%)": 55.45, "Format Valid (%)":  99.01, "BLEU-4":  5.78, "ROUGE-L": 38.86, "Latency (s)":  7.65, "Throughput (tok/s)":  6.63},
    {"Model Name": "Qwen2.5-3B-Base-CoT",     "Regime": "A (3-Shot Prompting)",     "Params": "3B",              "Tool Acc (%)": 62.38, "Macro F1 (%)": 26.15, "Arg Match (%)": 59.41, "Strict AST (%)": 59.41, "Format Valid (%)":  82.18, "BLEU-4":  5.16, "ROUGE-L": 30.11, "Latency (s)":  8.42, "Throughput (tok/s)":  6.92},
    {"Model Name": "Llama-3.2-3B",            "Regime": "A (Zero-Shot)",        "Params": "3B",              "Tool Acc (%)": 45.54, "Macro F1 (%)":  2.72, "Arg Match (%)": 45.54, "Strict AST (%)": 45.54, "Format Valid (%)":  45.54, "BLEU-4":  2.43, "ROUGE-L": 39.96, "Latency (s)": 20.97, "Throughput (tok/s)":  2.97},
    {"Model Name": "Phi-4-mini-3.8B",         "Regime": "A (Zero-Shot)",        "Params": "3.8B",            "Tool Acc (%)": 45.54, "Macro F1 (%)":  2.72, "Arg Match (%)": 45.54, "Strict AST (%)": 45.54, "Format Valid (%)":  45.54, "BLEU-4":  0.05, "ROUGE-L":  1.39, "Latency (s)":  7.19, "Throughput (tok/s)": 14.07},
    {"Model Name": "Gemma-4-26B-MoE",         "Regime": "A (Zero-Shot)",        "Params": "26B (3.8B Act)",  "Tool Acc (%)": 45.54, "Macro F1 (%)":  2.72, "Arg Match (%)": 45.54, "Strict AST (%)": 45.54, "Format Valid (%)":  45.54, "BLEU-4":  0.00, "ROUGE-L":  1.06, "Latency (s)":  1.13, "Throughput (tok/s)":  0.00},
    {"Model Name": "GLM-5.2",                 "Regime": "A (Zero-Shot)",        "Params": "18.6B Active",    "Tool Acc (%)": 45.54, "Macro F1 (%)":  2.72, "Arg Match (%)": 45.54, "Strict AST (%)": 45.54, "Format Valid (%)":  45.54, "BLEU-4":  0.13, "ROUGE-L":  1.95, "Latency (s)": 10.18, "Throughput (tok/s)":  0.72},
    {"Model Name": "MiniMax-M3",              "Regime": "A (Zero-Shot)",        "Params": "MoE Agentic",     "Tool Acc (%)": 45.54, "Macro F1 (%)":  2.72, "Arg Match (%)": 45.54, "Strict AST (%)": 45.54, "Format Valid (%)":  45.54, "BLEU-4": 24.63, "ROUGE-L": 48.69, "Latency (s)":  2.51, "Throughput (tok/s)": 35.92},
    {"Model Name": "Nemotron-3.5",            "Regime": "A (Zero-Shot)",        "Params": "30B-A3B",         "Tool Acc (%)": 45.54, "Macro F1 (%)":  2.72, "Arg Match (%)": 45.54, "Strict AST (%)": 45.54, "Format Valid (%)":  45.54, "BLEU-4":  0.00, "ROUGE-L":  1.45, "Latency (s)":  1.44, "Throughput (tok/s)":  0.00},
    {"Model Name": "Claude-Opus-5",           "Regime": "A (Zero-Shot)",        "Params": "Next-Gen Front.", "Tool Acc (%)": 45.54, "Macro F1 (%)":  2.72, "Arg Match (%)": 45.54, "Strict AST (%)": 45.54, "Format Valid (%)":  45.54, "BLEU-4": 18.62, "ROUGE-L": 44.88, "Latency (s)":  7.72, "Throughput (tok/s)": 14.01},
    {"Model Name": "GPT-5.6-Terra",           "Regime": "A (Zero-Shot)",        "Params": "Proprietary Cloud", "Tool Acc (%)": 45.54, "Macro F1 (%)":  2.72, "Arg Match (%)": 45.54, "Strict AST (%)": 45.54, "Format Valid (%)":  45.54, "BLEU-4":  0.00, "ROUGE-L":  0.41, "Latency (s)":  1.42, "Throughput (tok/s)":  0.00},
    {"Model Name": "GPT-4o",                  "Regime": "A (Zero-Shot)",        "Params": "Proprietary Cloud", "Tool Acc (%)": 45.54, "Macro F1 (%)":  2.72, "Arg Match (%)": 45.54, "Strict AST (%)": 45.54, "Format Valid (%)":  45.54, "BLEU-4":  0.00, "ROUGE-L":  0.17, "Latency (s)":  1.54, "Throughput (tok/s)":  0.00},
    {"Model Name": "Gemini-2.5-Flash",        "Regime": "A (Zero-Shot)",        "Params": "Cloud API",       "Tool Acc (%)": 45.54, "Macro F1 (%)":  2.72, "Arg Match (%)": 45.54, "Strict AST (%)": 45.54, "Format Valid (%)":  45.54, "BLEU-4":  0.00, "ROUGE-L":  0.04, "Latency (s)":  0.05, "Throughput (tok/s)":  0.00}
]

for entry in all_regimeB_summary:
    entry["Regime"] = "B (3-Shot Prompting (Regime B))"

all_combined = regime_a_results + all_regimeB_summary
df_combined = pd.DataFrame(all_combined)

col_order = ["Model Name", "Regime", "Params", "Tool Acc (%)", "Macro F1 (%)", "Arg Match (%)", "Strict AST (%)", "Format Valid (%)", "BLEU-4", "ROUGE-L", "Latency (s)", "Throughput (tok/s)"]
existing_cols = [c for c in col_order if c in df_combined.columns]
df_combined = df_combined[existing_cols]

print("="*140)
print("TABEL KOMPARASI HEAD-TO-HEAD: REGIME A (ZERO-SHOT) vs REGIME B (FEW-SHOT ENHANCED) -- BAB 4 SKRIPSI:")
print("="*140)
print(df_combined.to_string(index=False))
print("="*140)

COMBINED_CSV = os.path.join(BASE_OUTPUT_DIR, "RegimeAB_combined_benchmark_summary.csv")
df_combined.to_csv(COMBINED_CSV, index=False)
print(f"[OK] Tabel kombinasi tersimpan di: {COMBINED_CSV}")


## Section 7:  Grafik Komparasi Regime A vs Regime B (Delta Peningkatan)


In [ ]:
print("[*] Merender Grafik Komparasi Regime A vs Regime B...")

ra_dict = {}
for r in regime_a_results:
    key = r["Model Name"].replace("-3.2-3B-Instruct", "-3.2-3B").replace("-Reasoning", "").replace("-Agentic", "").replace("-Lightning", "").replace("-A4B-MoE", "-MoE")
    ra_dict[key] = r

rb_dict = {}
for r in all_regimeB_summary:
    key = r["Model Name"].replace("-FewShot", "")
    key = key.replace("-3.2-3B-Instruct", "-3.2-3B").replace("-4-26B", "-4-26B-MoE").replace("-3.5", "-3.5")
    rb_dict[key] = r

paired_models = []
ra_vals = []
rb_vals = []
for model_key, rb_data in rb_dict.items():
    for ra_key, ra_data in ra_dict.items():
        if model_key.split("-FewShot")[0].lower().replace(" ", "") in ra_key.lower().replace(" ", "") or ra_key.lower().replace(" ", "") in model_key.lower().replace(" ", ""):
            paired_models.append(ra_key)
            ra_vals.append(ra_data.get("Strict AST (%)", 45.54))
            rb_vals.append(rb_data.get("Strict AST (%)", 0))
            break

if paired_models:
    x = np.arange(len(paired_models))
    width = 0.35

    fig, ax = plt.subplots(figsize=(16, 7), dpi=300)
    bars_a = ax.bar(x - width/2, ra_vals, width, label="Regime A (Zero-Shot)", color="#FF6B6B", edgecolor="black", linewidth=0.5)
    bars_b = ax.bar(x + width/2, rb_vals, width, label="Regime B (3-Shot Prompting (Regime B))", color="#4ECDC4", edgecolor="black", linewidth=0.5)

    ax.axhline(y=80.20, color="gold", linestyle="--", linewidth=2, label="TelkomNusa-SLM-3B (SFT) = 80.20%")

    ax.set_ylabel("Strict AST Accuracy (%)", fontsize=12)
    ax.set_title("Komparasi Strict AST: Regime A (Zero-Shot) vs Regime B (3-Shot Prompting (Regime B))\nvs TelkomNusa-SLM-3B (Fine-Tuned SFT)", fontsize=13, fontweight="bold")
    ax.set_xticks(x)
    ax.set_xticklabels(paired_models, rotation=30, ha="right", fontsize=9)
    ax.legend(fontsize=10)
    ax.set_ylim(0, 110)
    ax.grid(axis="y", linestyle="--", alpha=0.5)

    for bar_a, bar_b in zip(bars_a, bars_b):
        ax.text(bar_a.get_x() + bar_a.get_width()/2, bar_a.get_height()+1, f"{bar_a.get_height():.1f}%", ha="center", va="bottom", fontsize=8)
        ax.text(bar_b.get_x() + bar_b.get_width()/2, bar_b.get_height()+1, f"{bar_b.get_height():.1f}%", ha="center", va="bottom", fontsize=8, fontweight="bold")

    plt.tight_layout()
    CHART_AB = os.path.join(BASE_OUTPUT_DIR, "regimeAB_comparison_chart.png")
    plt.savefig(CHART_AB, bbox_inches="tight")
    plt.show()
    print(f"[OK] Grafik Regime A vs B tersimpan di: {CHART_AB}")
else:
    print("[INFO] Tidak ada pasangan model yang bisa dipasangkan untuk grafik.")


## Section 8:  Download Semua Hasil Regime B


In [ ]:
import zipfile
from IPython.display import HTML, display, FileLink

ZIP_NAME = "Hasil_Evaluasi_RegimeB_FewShot_Enhanced.zip"
ZIP_PATH = os.path.join(BASE_OUTPUT_DIR, ZIP_NAME)
VALID_EXT = ('.csv', '.json', '.png')

files_to_zip = []
for root, dirs, files in os.walk(BASE_OUTPUT_DIR):
    for f in files:
        if f.startswith("RegimeB") or f.startswith("regimeAB"):
            if f.endswith(VALID_EXT):
                files_to_zip.append(os.path.join(root, f))

combined_csv = os.path.join(BASE_OUTPUT_DIR, "RegimeAB_combined_benchmark_summary.csv")
if os.path.exists(combined_csv) and combined_csv not in files_to_zip:
    files_to_zip.append(combined_csv)

if files_to_zip:
    with zipfile.ZipFile(ZIP_PATH, 'w', zipfile.ZIP_DEFLATED) as zipf:
        for fp in files_to_zip:
            zipf.write(fp, arcname=os.path.basename(fp))
            print(f"[+] {os.path.basename(fp):55s} ({os.path.getsize(fp)/1024:6.1f} KB)")

    print(f"\n[OK] {len(files_to_zip)} file dipaketkan ke: {ZIP_NAME} ({os.path.getsize(ZIP_PATH)/(1024*1024):.2f} MB)")
    display(FileLink(ZIP_NAME))
    try:
        from google.colab import files
        files.download(ZIP_PATH)
    except: pass
else:
    print("[!] Tidak ada file Regime B untuk di-zip.")
